# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/dipson-mishra/flyrank-ml/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

In [1]:
import os
from pathlib import Path
import duckdb
import pandas as pd

def load_hf_token():
    for path in [Path.cwd() / '.env', Path.cwd().parent / '.env', Path.cwd().parent.parent / '.env']:
        if path.exists():
            for line in path.read_text().splitlines():
                if line.strip().startswith('HF_TOKEN='):
                    return line.split('=', 1)[1].strip().strip('\"').strip(chr(39))
    return os.environ.get('HF_TOKEN')

token = load_hf_token()
if not token:
    raise RuntimeError('HF_TOKEN required via .env or the environment.')
con = duckdb.connect()
con.execute('CREATE OR REPLACE SECRET hf (TYPE HUGGINGFACE, TOKEN ?)', [token])
FACT = 'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
CONTENT = 'hf://datasets/FlyRank/internship-warehouse/dim_content.parquet'
query = """
WITH daily AS (
  SELECT content_hash_id, client_hash_id,
         SUM(COALESCE(gsc_impressions, 0)) AS impressions,
         SUM(COALESCE(gsc_clicks, 0)) AS clicks,
         SUM(COALESCE(gsc_sum_position, 0)) AS sum_position,
         SUM(COALESCE(ga4_sessions, 0)) AS sessions,
         SUM(COALESCE(ga4_engaged_sessions, 0)) AS engaged_sessions,
         SUM(COALESCE(scroll_events, 0)) AS scroll_events,
         SUM(COALESCE(ga4_pageviews, 0)) AS pageviews,
         SUM(CASE WHEN report_date >= '2026-03-16' THEN COALESCE(gsc_clicks, 0) ELSE 0 END) AS clicks_h2,
         SUM(CASE WHEN report_date < '2026-03-16' THEN COALESCE(gsc_clicks, 0) ELSE 0 END) AS clicks_h1
  FROM read_parquet(?)
  WHERE gsc_data_available IS TRUE
  GROUP BY content_hash_id, client_hash_id
)
SELECT d.*, c.content_type, c.main_intent, c.word_count,
       d.clicks * 100.0 / NULLIF(d.impressions, 0) AS ctr,
       d.sum_position * 1.0 / NULLIF(d.impressions, 0) AS avg_position,
       CASE WHEN d.clicks_h2 < d.clicks_h1 THEN 1 ELSE 0 END AS is_declining_label
FROM daily d
LEFT JOIN read_parquet(?) c USING (content_hash_id)
WHERE d.impressions >= 500
"""
df = con.execute(query, [FACT, CONTENT]).df()
print('rows,cols:', df.shape)
print('label counts:\n', df['is_declining_label'].value_counts(dropna=False))
df.head(3)


rows,cols: (61924, 17)
label counts:
 is_declining_label
0    40047
1    21877
Name: count, dtype: int64


,content_hash_id,client_hash_id,impressions,clicks,sum_position,sessions,engaged_sessions,scroll_events,pageviews,clicks_h2,clicks_h1,content_type,main_intent,word_count,ctr,avg_position,is_declining_label
0,content_077056ca9902e54b,client_62f4a7e64f5e0096,1304.0,4.0,1840.0,0.0,0.0,0.0,0.0,4.0,0.0,keyword article,transactional,<NA>,0.306748,1.411043,0
1,content_18b6dbed1447e8b5,client_62f4a7e64f5e0096,19647.0,79.0,51313.0,0.0,0.0,0.0,0.0,50.0,29.0,keyword article,transactional,<NA>,0.402097,2.611747,0
2,content_901cd406c3099ffd,client_62f4a7e64f5e0096,2579.0,3.0,9104.0,0.0,0.0,0.0,0.0,1.0,2.0,keyword article,informational,2771,0.116324,3.530050,1


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

In [2]:
from sklearn.model_selection import GroupShuffleSplit
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_ix, test_ix = next(gss.split(df, groups=df['client_hash_id']))
print('train rows, test rows:', len(train_ix), len(test_ix))
print('train label rate:', df.loc[train_ix, 'is_declining_label'].mean())
print('test label rate:', df.loc[test_ix, 'is_declining_label'].mean())
per_client = df.groupby('client_hash_id')['is_declining_label'].agg(['mean','count']).rename(columns={'mean':'rate','count':'n'})
per_client.sort_values('n', ascending=False).head()


train rows, test rows: 43700 18224
train label rate: 0.3577574370709382
test label rate: 0.3425702370500439


,rate,n
client_hash_id,,
client_73cda7b4e4f265ea,0.384507,14252
client_62f4a7e64f5e0096,0.393959,11090
client_23a62021009f63c4,0.389039,9999
client_e547b89c05043229,0.258336,6058
client_fef1a8f436438636,0.332507,5245


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

In [3]:
# Leakage probes
num = df.select_dtypes(include=['number']).copy()
if 'is_declining_label' in num.columns:
    num = num.drop(columns=['is_declining_label'])
corrs = num.corrwith(df['is_declining_label']).abs().sort_values(ascending=False)
print('Top numeric correlations with label (abs):')
print(corrs.head(10))
bucket = df.groupby('content_type')['is_declining_label'].agg(['mean','count']).rename(columns={'mean':'rate'})
print('content_type rates (showing n):')
print(bucket.sort_values('count', ascending=False).head(10))


Top numeric correlations with label (abs):
clicks_h1           0.109383
impressions         0.074684
sum_position        0.063209
ctr                 0.061872
avg_position        0.051488
clicks_h2           0.046432
sessions            0.044273
pageviews           0.042452
scroll_events       0.036246
engaged_sessions    0.028601
dtype: float64
content_type rates (showing n):
                        rate  count
content_type                       
keyword article     0.354209  61475
feedly article      0.267974    306
comparison article  0.139860    143


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

In [4]:
# Claim rewrite placeholder — edit the markdown cell above with a careful claim.
print('Please write the claim in the markdown cell above before submitting.')


Please write the claim in the markdown cell above before submitting.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.